In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("/home/users/mm2947/jupyter/DMD/dmd_qbo")))
#from dmd_functions import *
from dmd_functions_adapted import *
import numpy as np
import pygeode as pyg
from matplotlib import pyplot as plt
import xarray as xr
from pydmd import EDMD
from pydmd.preprocessing import hankel_preprocessing
from scipy.signal import detrend
import pandas as pd
from matplotlib.offsetbox import AnchoredText
import textwrap
from math import ceil

%matplotlib widget

# **Data loading & settings**

In [ ]:
# ----------------------------
# Load data
# ----------------------------
model = "E3SM_PDINT"
file_name = "E3SM_PDINT_anom_smooth_1"
data_file_out = '/home/users/mm2947/jupyter/processed_data/E3SM/' + file_name + '.nc'
ds = xr.open_dataset(data_file_out)
print(ds)

In [ ]:
# ----------------------------
# Data settings
# ----------------------------
pres_min = ds.pres.values.min()       # hPa
pres_max = ds.pres.values.max()       # hPa
lat_min = ds.lat.values.min()
lat_max = ds.lat.values.max()
lat_band = (-10, 10)
#lat_band = (FT_lat_band.start, FT_lat_band.stop)
lat_min = get_index(lat_band[0], ds.lat.values)
lat_max = get_index(lat_band[1], ds.lat.values)

# ----------------------------
# Convenience definitions
# ----------------------------
#input_vars = ["u"] 
input_vars = ["u", "T"] 
var_labels = {"u": "Zonal wind (u)", "T": "Temperature (T)"}
var_units = {"u": "m/s", "T": "K"}
u_levs = np.linspace(-20, 20, 11)
T_levs = np.linspace(-1.4, 1.4, 11)
pres_plot = 50.0

# ----------------------------
# FT settings
# ----------------------------
FT_pres_band = (10.0, 100.0)    
FT_lat_band = slice(*lat_band)
xlim_months = (0, 50)
n_peaks = 3

# ----------------------------
# DMD settings
# ----------------------------
stab_threshold = 0.01
d_multipliers = np.arange(1, 2.01, 0.5)
dt = 1.0

# ----------------------------
# Spectral entropy rank truncation settings
# ----------------------------
alpha = 0.001   # 0.1% threshold

# ----------------------------
# Output settings
# ----------------------------
save = True
save_dir = '/home/users/mm2947/jupyter/output'

# **Fourier transform to extract target QBO mode period**

In [ ]:
ft_results = compute_fft_summary(
    ds,
    input_vars=input_vars,
    p_band=FT_pres_band,
    lat_band=FT_lat_band,
    dt=dt,
    xlim=xlim_months,
    n_peaks=n_peaks,
    var_labels=var_labels,
    var_units=var_units,
    make_plots=True,
)

qbo_period = ft_results['u']['peaks'][0]
print(f'QBO period extracted from fourier transform = {qbo_period}')
qbo_band = (qbo_period - 2*dt, qbo_period + 2*dt)

# **Data matrix assembling**

In [ ]:
# ----------------------------
# Standardise and form data matrix X
# ----------------------------

std_option = 3
stds = []
std_data = []

for v in input_vars:
    data = ds[v]
    
    if std_option == 0:
        # Height-dependent std
        std = 1.0 / ds[v].std(axis=0, ddof=1).values
    elif std_option == 1:
        # Scalar std
        std = 1.0 / ds[v].std(ddof=1).values
    elif std_option == 2:
        # Variance-normalisation
        std = 1.0 / np.sqrt(np.var(np.asarray(ds[v]), axis=0) + 1e-8)
    elif std_option == 3:
        # density & area weighting
        lat_w = np.cos(np.deg2rad(ds.lat.values))  
        
        # dp weight (proportional to mass per unit area)
        p = ds.pres.values
        dp = np.empty_like(p)
        dp[1:-1] = 0.5 * (p[2:] - p[:-2])
        dp[0] = p[1] - p[0]
        dp[-1] = p[-1] - p[-2]
        dp = np.abs(dp)
        
        # combined grid weight w(lat,p) proportional to area * thickness
        w = lat_w[None, :] * dp[:, None]                   # (nlat, npres)

        std = w / ds[v].std(ddof=1).values

    stds.append(std)
    std_data.append((ds[v] * std.T).T.values)

# Arrange into DMD snapshot matrix X
X = np.stack(std_data, axis=0)
dshape = X.shape
print(X.shape)

n_time = X.shape[-1]
assert n_time == len(ds.time)
X2d = X.reshape(-1, n_time)
nfeatures, ntime = X2d.shape
print(X2d.shape)

In [ ]:
# ----------------------------
# Conditioning checks
# ----------------------------

print("dtype:", X2d.dtype, "shape:", X2d.shape)
print("NaNs:", np.isnan(X2d).sum(), "infs:", np.isinf(X2d).sum())
print("finite fraction:", np.isfinite(X2d).mean())

bad = ~np.isfinite(X2d)
print("Bad rows:", np.sum(bad.any(axis=1)), "Bad cols:", np.sum(bad.any(axis=0)))

In [ ]:
# ----------------------------
# Visualise data via tropical mean (WRAPPED)
# ----------------------------

def compute_points_per_row(nt, target_rows=2):
    return max(1, nt // target_rows)

for i, var in enumerate(std_data):

    var = (var.T / stds[i].T).T
    name = input_vars[i]
    levs = u_levs if name == "u" else T_levs
    
    # Assumes latitude is second axis
    trop_mean = np.mean(var[:, lat_min:lat_max], axis=1)
    nt = trop_mean.shape[1]

    points_per_row = compute_points_per_row(nt)
    nrows = int(np.ceil(nt / points_per_row))

    fig, axs = plt.subplots(
        nrows, 1,
        figsize=(14, 2.2 * nrows),
        sharey=True,
        constrained_layout=True
    )
    axs = np.atleast_1d(axs)

    for r in range(nrows):
        i0 = r * points_per_row
        i1 = min((r + 1) * points_per_row, nt)

        x = np.arange(i0, i1)        # numeric time index
        Z = trop_mean[:, i0:i1]

        ax = axs[r]
        im = ax.contourf(
            x, ds.pres.values, Z,
            levels=levs, cmap='bwr', extend='both'
        )

        # ax.set_yscale('log')
        # ax.invert_yaxis()
        ax.axhline(70.0, color='k', ls='--', lw=1.0)
        ax.grid(True, axis='x', alpha=0.3)

        # label each row so it's obvious what's happening
        ax.set_title(f"{input_vars[i]} | time index {i0}–{i1}", loc='left', fontsize=10)

        if r < nrows - 1:
            ax.set_xlabel("")
        else:
            ax.set_xlabel("Time index")

    for ax in axs:
        ax.set_yscale('log')

    axs[0].invert_yaxis()   # only once, on the master
    
    fig.colorbar(im, ax=axs, location='bottom')
    plt.show()


# **Kernel patching**

In [ ]:
# ----------------------------
# NLSA kernel (Szekely et al., 2016)
# ----------------------------

def nlsa_kernel(X, Y=None, *, epsilon=None, speed_floor=1e-12):
    """
    Székely et al. (2016) eq.(2):

      K_ij = exp( -||x_i - y_j||^2 / (epsilon * ||zeta_i|| * ||zeta_j||) )
      zeta_i = x_i - x_{i-1} is the phase-space velocity  (rows time-ordered within each array)

    Returns: (nX, nY) ndarray.
    """
    
    same = False
    if Y is None:
        Y = X
        same = True

    if epsilon is None:
        epsilon = 1.0
    if epsilon <= 0:
        raise ValueError("epsilon must be positive.")

    def phase_space_velocity(A):
        n = A.shape[0]
        if n == 0:
            return np.empty((0,), dtype=float)
        if n == 1:
            return np.ones((1,), dtype=float)
        dA = np.empty_like(A)
        dA[1:] = A[1:] - A[:-1]  # backward difference
        dA[0] = dA[1]            # boundary fix
        v = np.linalg.norm(dA, axis=1)
        return np.maximum(v, speed_floor)

    vx = phase_space_velocity(X)
    vy = vx if same else phase_space_velocity(Y)

    # squared distances: ||x-y||^2 = ||x||^2 + ||y||^2 - 2 x·y
    # could be done using linalg.norm but this is probably faster
    x2 = np.sum(X * X, axis=1)
    y2 = np.sum(Y * Y, axis=1)
    G = X @ Y.T
    d2 = x2[:, None] + y2[None, :] - 2.0 * G
    d2 = np.maximum(d2, 0.0)

    denom = epsilon * (vx[:, None] * vy[None, :])
    denom = np.maximum(denom, 1e-12) # prevent div by 0 error

    return np.exp(-d2 / denom)

# Patch eDMD
import pydmd.edmd as edmd_mod
_ORIG_PAIRWISE_KERNELS = edmd_mod.pairwise_kernels

def _pairwise_kernels_fast(X, Y=None, metric="linear", filter_params=False, n_jobs=None, **kwds):
    """
    Replacement for sklearn.metrics.pairwise.pairwise_kernels that
    allows custom vectorized kernel callables metric(X, Y, **kwds).
    """
    # Follow sklearn behavior: if Y is None, use Y=X (unless precomputed)
    if Y is None:
        Y = X

    # Modified behaviour: if metric is callable, try vectorized kernel call
    if callable(metric):
        try:
            K = metric(X, Y, **kwds)
            K = np.asarray(K)
            if K.ndim == 2 and K.shape == (X.shape[0], Y.shape[0]):
                return K
        except TypeError:
            # Likely a scalar metric k(x,y); fall back
            pass

    # Fall back to original sklearn-based implementation used by PyDMD
    return _ORIG_PAIRWISE_KERNELS(X, Y, metric=metric, filter_params=filter_params, n_jobs=n_jobs, **kwds)

# Apply patch inside PyDMD
edmd_mod.pairwise_kernels = _pairwise_kernels_fast

# **Delay embed length selection**

In [ ]:
eigenstructure = {}
X_base = X2d.copy()
n_features = X_base.shape[0]
    
eigenstructure, d_list = compute_eigenstructure_over_delays(
    X2d=X2d,
    stab_threshold=stab_threshold,
    d_multipliers=d_multipliers,
    qbo_period = qbo_period,
    nlsa_kernel = nlsa_kernel,
    dt = dt,
    qbo_band = qbo_band,
)

In [ ]:
best_d = select_delay_length(
    eigenstructure,
    d_list,
    qbo_period
)

In [ ]:
d_mid, sims = plot_mode_stability_across_d(eigenstructure, d_list)

# **Spectral entropy rank selection**

In [ ]:
# ----------------------------
# Preliminary fit to set svd_rank
# ----------------------------
d = int(best_d)
# Delay embedding 
H = hankelize(X2d, d)

edmd = EDMD(svd_rank=-1,
           kernel_metric=nlsa_kernel,
           kernel_params={"epsilon": 2.0}
           ).fit(H)

In [ ]:
#fig, svd_rank = plot_svd_spectrum(edmd, 2/3)
# print(f"{svd_rank} ranks required to achieve energy threshold.")

svalues = edmd.operator.svd_vals
spectral_entropy = compute_spectral_entropy(svalues, crop_n=50, make_plots=True)

In [ ]:
svd_rank = spectral_entropy_rank_truncation(spectral_entropy, alpha=alpha)

# **DMD fitting**

In [ ]:
# ----------------------------
# Fit DMD
# ----------------------------
dmd = EDMD(svd_rank=svd_rank,
           kernel_metric=nlsa_kernel,
           kernel_params={"epsilon": 2.0}
          )

dmd.fit(H)

dmd.original_time['dt'] = 1.0

In [ ]:
fig = plot_dmd_summary(dmd, dshape, d, qbo_band, qbo_period)

In [ ]:
lam = np.asarray(dmd.eigs)
b = dmd.amplitudes
modes = dmd.modes
dynamics = dmd.dynamics

theta = np.angle(lam)
omega = np.log(lam) / dt
freq = np.abs(theta) / (2*np.pi*dt)
period = np.where(freq > 0, 1.0/freq, np.inf)
print(np.sort(period))
amps = np.abs(b)

# **DMD reconstruction from all stable modes in QBO band**

In [ ]:
in_period = (period > qbo_band[0]) & (period < qbo_band[1])
print(f'Found {np.count_nonzero(in_period)} modes in the QBO band')
print(np.unique(period[in_period]))

keep_QBO = (period > qbo_band[0]) & (period < qbo_band[1]) & (np.abs(np.abs(lam) - 1.0) < stab_threshold)
dim_keep = np.count_nonzero(keep_QBO)
print(f'Found {dim_keep} stable modes in the QBO band')
print(np.unique(period[keep_QBO]))

if dim_keep == 0:
    print("No stable QBO modes found — skipping reconstruction")
    sys.exit()

else:    
    fig, out = plot_reconstruction_summary3(
        dmd, keep_QBO, d, dshape, ds, std_data, stds,
        u_levs, T_levs,
        var_labels = input_vars,
        qbo_band=qbo_band,
        lat_band=lat_band,
        pres_level=pres_plot,
        title="Stable modes in QBO band",
        return_outputs=True
    )


# **DMD reconstruction from single dominant QBO mode & its harmonics**

In [ ]:
p0 = period[np.argmin(np.abs(period - qbo_period))]
print(f'Identified dominant QBO mode has period {p0}')

keep_harm, harm = get_harmonics(p0, period, sampfreq=1.0)
keep_stable = np.abs(np.abs(lam) - 1.0) < stab_threshold
remove_annual = (period < 12.0 - dt) | (period > 12.0 + dt) # Avoids harmonics too close to the annual cycle
keep_QBO_harm = keep_harm & keep_stable & remove_annual
print(f'Harmonics of this dominant mode are {harm}')
print(f'Modes retained after filtering for stability and harmonic-like periodicity are {period[keep_QBO_harm]}')

fig, out_harm = plot_reconstruction_summary3(
    dmd, keep_QBO_harm, d, dshape, ds, std_data, stds,
    u_levs, T_levs,
    var_labels = input_vars,
    qbo_band = qbo_band,
    lat_band=lat_band,
    pres_level=pres_plot,
    title=f"Harmonics of main QBO mode with period {p0:.2f} months",
    return_outputs=True
)

# **DMD reconstruction from modes in QBO range & their harmonics**

In [ ]:
p0s = np.unique(period[(period > qbo_band[0]) & (period < qbo_band[1])])
print(p0s)

sf = 1.0
keep, harm = get_harmonics(p0s[0], period, sampfreq=sf)
if len(p0s) > 1:
    for p0 in p0s[1:]:
        keepnew, harmnew = get_harmonics(p0, period, sampfreq=1.0)
        keep |= keepnew
        harm = np.append(harm, harmnew)

# Stability mask
keep_stable = np.abs(np.abs(lam) - 1.0) < stab_threshold
remove_annual = (period < 12.0 - dt) | (period > 12.0 + dt) # Avoids harmonics too close to the annual cycle
keep_QBO_band_harm = keep & keep_stable & remove_annual
harm = np.unique(harm)
        
print(harm)
print(period[keep])
    
fig, out_band_harm = plot_reconstruction_summary3(
    dmd, keep_QBO_band_harm, d, dshape, ds, std_data, stds,
    u_levs, T_levs,
    var_labels = input_vars,
    qbo_band = qbo_band,
    lat_band=lat_band,
    pres_level=pres_plot,
    title=f"Harmonics of modes in QBO band {qbo_band[0]:.1f}-{qbo_band[1]:.1f} months",
    return_outputs=True
)


if save:
    #save_fig(fig, save_dir+"qbo_"+config_str, dpi=300)
    saving_name = f"{file_name}_vars={','.join(input_vars)}_d={d}.png"
    fig.savefig(
    os.path.join(save_dir, saving_name),
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
fig, _ = plot_reconstruction_summary3_no_annual(
    dmd, keep_QBO, d, dshape, ds, std_data, stds,
    u_levs, T_levs,
    var_labels=input_vars,
    title="Stable modes in QBO band (T original de-seasonalised)",
    return_outputs=True
)

# **Lat-Height Plots**

In [ ]:
def plot_lat_height(u, T, tidx, title):

    fig, axs = plt.subplots(1,2, figsize=(12,5), constrained_layout=True)

    im0 = axs[0].contourf(lat, pres, u[:,:,tidx], levels=u_levs, cmap="bwr", extend="both")
    axs[0].set_title("u reconstruction")
    axs[0].set_xlabel("Latitude")
    axs[0].set_ylabel("Pressure (hPa)")
    axs[0].set_yscale("log")
    axs[0].invert_yaxis()
    plt.colorbar(im0, ax=axs[0])

    im1 = axs[1].contourf(lat, pres, T[:,:,tidx], levels=T_levs, cmap="bwr", extend="both")
    axs[1].set_title("T reconstruction")
    axs[1].set_xlabel("Latitude")
    axs[1].set_yscale("log")
    axs[1].invert_yaxis()
    plt.colorbar(im1, ax=axs[1])

    fig.suptitle(f"{title} (t = {time[tidx]:.1f} months)")
    plt.show()


In [ ]:
# # Extract reconstructed fields
# X_rec = out_band_harm["X_rec_phys"]
# u_rec = X_rec[0]
# T_rec = X_rec[1]

# lat = ds.lat.values
# pres = ds.pres.values
# time = out_band_harm["tplot"]

# # Find time indices of max and min u
# tmax = np.argmax(np.max(u_rec, axis=(0,1)))
# tmin = np.argmin(np.min(u_rec, axis=(0,1)))

# print("Max u time:", time[tmax])
# print("Min u time:", time[tmin])

# plot_lat_height(u_rec, T_rec, tmax, "Maximum reconstructed u")

# plot_lat_height(u_rec, T_rec, tmin, "Minimum reconstructed u")
